# IGV notebook demo

This notebook uses `igv-notebook` to embed igv.js in a VS Code Jupyter notebook. It loads a local BAM with a BAI index and a reference FASTA with a FAI index.

## Requirements

- `pip install igv-notebook` in the active kernel environment
- Ensure the `.bai` and `.fai` index files exist alongside the BAM and FASTA files

In [ ]:
import igv_notebook

igv_notebook.init()

In [ ]:
# Paths from run.sh
BAM_FILE = "/home/peterkad/pkadmaster/data/mutationalscanning_bam/ph/diploid_assembly/ph_plus_unmapped_diploid_v2.bam"
FASTA_FILE = "/home/peterkad/pkadmaster/data/ph/ph_diploid.fa"

In [ ]:
import os
from pathlib import Path

BAM_INDEX = BAM_FILE + ".bai"
FASTA_INDEX = FASTA_FILE + ".fai"

for label, path in {
    "BAM": BAM_FILE,
    "BAM_INDEX": BAM_INDEX,
    "FASTA": FASTA_FILE,
    "FASTA_INDEX": FASTA_INDEX,
}.items():
    p = Path(path)
    exists = p.exists()
    size = p.stat().st_size if exists else None
    readable = os.access(p, os.R_OK) if exists else False
    print(f"{label}: exists={exists}, readable={readable}, size={size}, path={p}")

# Try opening a small chunk to confirm read access
with open(BAM_FILE, "rb") as bam_handle:
    bam_handle.read(64)
with open(FASTA_FILE, "rb") as fasta_handle:
    fasta_handle.read(64)

print("Basic read checks passed.")

In [ ]:
# Pick a contig from the FASTA index so the locus exists
with open(FASTA_INDEX, "r", encoding="utf-8") as fai_handle:
    first_line = fai_handle.readline().strip()

if not first_line:
    raise ValueError("FASTA index is empty.")

contig_name = first_line.split("\t", 1)[0]
locus = f"{contig_name}:1-1000"
print(f"Using locus: {locus}")

In [ ]:
from IPython.display import display

igv_browser = igv_notebook.Browser(
    {
        "genome": "custom",
        "reference": {
            "id": "custom",
            "name": "custom",
            "fastaPath": FASTA_FILE,
            "indexPath": FASTA_INDEX,
        },
        "locus": locus,
    }
)

display(igv_browser)

In [ ]:
# Load the BAM track after the browser is displayed
igv_browser.load_track(
    {
        "name": "BAM",
        "path": BAM_FILE,
        "indexPath": BAM_INDEX,
        "format": "bam",
        "type": "alignment",
    }
)

# Navigate to a known contig region
igv_browser.search(locus)

## Read depth + QC (sperm long reads)

This section estimates read depth, aligned bases, MAPQ distribution, and simple duplicate heuristics.
It uses the same `BAM_FILE` / `FASTA_FILE` variables defined above.

In [ ]:
import pysam
import numpy as np
import pandas as pd
from collections import Counter

# Optional: set to an integer to subsample reads for quick checks
MAX_READS = None  # e.g., 200000

bam = pysam.AlignmentFile(BAM_FILE, "rb")
fasta = pysam.FastaFile(FASTA_FILE)

ref_lengths = {name: fasta.get_reference_length(name) for name in fasta.references}
ref_total_len = sum(ref_lengths.values())
print(f"Reference total length: {ref_total_len:,} bp")

In [ ]:
def aligned_bases_from_cigar(cigartuples):
    if cigartuples is None:
        return 0
    # ops that consume reference
    ref_ops = {0, 2, 3, 7, 8}  # M, D, N, =, X
    return sum(length for op, length in cigartuples if op in ref_ops)

stats = {
    "total_reads": 0,
    "mapped_reads": 0,
    "primary_reads": 0,
    "secondary": 0,
    "supplementary": 0,
    "aligned_bases": 0,
}

read_lengths = []
mapqs = []
per_contig_aligned = Counter()

for idx, read in enumerate(bam.fetch(until_eof=True)):
    if MAX_READS and idx >= MAX_READS:
        break
    stats["total_reads"] += 1
    if read.is_secondary:
        stats["secondary"] += 1
    if read.is_supplementary:
        stats["supplementary"] += 1
    if read.is_unmapped:
        continue
    stats["mapped_reads"] += 1
    if not read.is_secondary and not read.is_supplementary:
        stats["primary_reads"] += 1
    read_lengths.append(read.query_length or 0)
    mapqs.append(read.mapping_quality)
    aln_bases = aligned_bases_from_cigar(read.cigartuples)
    stats["aligned_bases"] += aln_bases
    if read.reference_name:
        per_contig_aligned[read.reference_name] += aln_bases

stats

In [ ]:
mean_depth = stats["aligned_bases"] / ref_total_len if ref_total_len else 0
print(f"Mean depth (aligned bases / ref length): {mean_depth:.3f}x")

read_lengths_arr = np.array(read_lengths) if read_lengths else np.array([0])
print(f"Read length median: {np.median(read_lengths_arr):.0f}")

# N50 calculation
sorted_lengths = np.sort(read_lengths_arr)[::-1]
cum = np.cumsum(sorted_lengths)
if cum[-1] > 0:
    n50 = sorted_lengths[np.searchsorted(cum, cum[-1] / 2)]
else:
    n50 = 0
print(f"Read length N50: {n50:.0f}")

per_contig = (
    pd.DataFrame(
        [(c, per_contig_aligned[c], ref_lengths.get(c, 0)) for c in per_contig_aligned],
        columns=["contig", "aligned_bases", "contig_len"],
    )
    .assign(mean_depth=lambda df: df.aligned_bases / df.contig_len)
    .sort_values("mean_depth", ascending=False)
)
per_contig.head(10)

In [ ]:
import matplotlib.pyplot as plt

plt.hist(mapqs, bins=50)
plt.title("MAPQ distribution")
plt.xlabel("MAPQ")
plt.ylabel("Reads")
plt.show()

In [ ]:
# Simple duplicate heuristics
# 1) duplicate read names
# 2) duplicate alignment signatures (contig,start,end,cigar)

bam.reset()
read_name_counts = Counter()
aln_sig_counts = Counter()

for idx, read in enumerate(bam.fetch(until_eof=True)):
    if MAX_READS and idx >= MAX_READS:
        break
    read_name_counts[read.query_name] += 1
    if read.is_unmapped or read.reference_name is None:
        continue
    aln_sig = (
        read.reference_name,
        read.reference_start,
        read.reference_end,
        read.cigarstring,
    )
    aln_sig_counts[aln_sig] += 1

name_dupes = sum(1 for v in read_name_counts.values() if v > 1)
aln_dupes = sum(1 for v in aln_sig_counts.values() if v > 1)

print(f"Duplicate read names: {name_dupes} of {len(read_name_counts)}")
print(f"Duplicate alignment signatures: {aln_dupes} of {len(aln_sig_counts)}")

## Multi-sample reporting (TR / PH / Tsaed)

This section loads the latest run for each sample under the standard results layout, computes summary statistics, and generates publication-ready plots and tables.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# ---- Configure these ----
results_root = Path("/home/peterkad/pkadmaster/indel_scanner/results")
samples = ["TR", "PH", "Tsaed"]

figures_dir = Path("figures")
tables_dir = Path("tables")
figures_dir.mkdir(exist_ok=True)
tables_dir.mkdir(exist_ok=True)

sns.set_theme(style="whitegrid")

In [ ]:
def latest_run_dir(sample_root: Path) -> Path | None:
    if not sample_root.exists():
        return None
    run_dirs = [p for p in sample_root.iterdir() if p.is_dir()]
    if not run_dirs:
        return None
    return max(run_dirs, key=lambda p: p.stat().st_mtime)

records = []
missing = []

for sample in samples:
    sample_root = results_root / sample
    run_dir = latest_run_dir(sample_root)
    if run_dir is None:
        missing.append((sample, "no_run_dir"))
        continue
    per_type = run_dir / "per_type_mutation_frequency.tsv"
    callable_bases = run_dir / "callable_bases.tsv"
    if not per_type.exists() or not callable_bases.exists():
        missing.append((sample, str(run_dir)))
        continue
    records.append(
        {
            "sample": sample,
            "run_dir": run_dir,
            "per_type": per_type,
            "callable_bases": callable_bases,
        }
    )

pd.DataFrame(records), pd.DataFrame(missing, columns=["sample", "issue"])

In [ ]:
def load_per_type(path: Path, sample: str) -> pd.DataFrame:
    df = pd.read_csv(path, sep="\t")
    df["sample"] = sample
    df["indel_type"] = df["mutation_type"].str.split("_").str[0]
    df["len_or_motif"] = (
        df["mutation_type"]
        .str.extract(r"_(?:len|motif)_(\d+)bp")[0]
        .astype(float)
    )
    return df

per_type_dfs = []
callable_dfs = []

for rec in records:
    sample = rec["sample"]
    per_type_dfs.append(load_per_type(rec["per_type"], sample))
    callable_df = pd.read_csv(rec["callable_bases"], sep="\t")
    callable_df["sample"] = sample
    callable_dfs.append(callable_df)

per_type_all = pd.concat(per_type_dfs, ignore_index=True) if per_type_dfs else pd.DataFrame()
callable_all = pd.concat(callable_dfs, ignore_index=True) if callable_dfs else pd.DataFrame()

per_type_all.head()

In [ ]:
def summarize_rates(df: pd.DataFrame) -> pd.DataFrame:
    summary = []
    for (sample, str_class), grp in df.groupby(["sample", "str_class"]):
        total_count = grp["count"].sum()
        total_unique = grp["unique_sites"].sum()
        total_callable = grp["callable_bases"].sum()
        rate = total_count / total_callable if total_callable > 0 else np.nan
        unique_rate = total_unique / total_callable if total_callable > 0 else np.nan
        summary.append(
            {
                "sample": sample,
                "str_class": str_class,
                "total_count": total_count,
                "total_unique": total_unique,
                "total_callable": total_callable,
                "rate": rate,
                "unique_rate": unique_rate,
            }
        )
    return pd.DataFrame(summary)

summary_df = summarize_rates(per_type_all)
summary_df

In [ ]:
summary_df.to_csv(tables_dir / "summary_rates.csv", index=False)
per_type_all.to_csv(tables_dir / "per_type_all.csv", index=False)
callable_all.to_csv(tables_dir / "callable_bases_all.csv", index=False)

In [ ]:
# STR vs non-STR rate per sample
plt.figure(figsize=(6, 4))
ax = sns.pointplot(
    data=summary_df,
    x="sample",
    y="rate",
    hue="str_class",
    dodge=0.3,
    markers="o",
)
ax.set_ylabel("Mutation rate (per base)")
ax.set_xlabel("Sample")
ax.set_yscale("log")
ax.set_title("STR vs non-STR mutation rates")
plt.tight_layout()
plt.savefig(figures_dir / "rates_str_vs_non_str.png", dpi=300)
plt.savefig(figures_dir / "rates_str_vs_non_str.pdf")
plt.show()

In [ ]:
# Length / motif spectra
spectra = per_type_all.dropna(subset=["len_or_motif"]).copy()
spectra["len_or_motif"] = spectra["len_or_motif"].astype(int)

# Non-STR length spectrum
non_str = spectra[spectra["str_class"] == "non_STR"]
plt.figure(figsize=(7, 4))
ax = sns.lineplot(
    data=non_str,
    x="len_or_motif",
    y="count",
    hue="sample",
    style="indel_type",
    markers=True,
)
ax.set_xlabel("Indel length (bp)")
ax.set_ylabel("Count")
ax.set_title("Non-STR indel length spectrum")
plt.tight_layout()
plt.savefig(figures_dir / "spectrum_non_str.png", dpi=300)
plt.savefig(figures_dir / "spectrum_non_str.pdf")
plt.show()

# STR motif spectrum
str_df = spectra[spectra["str_class"] == "STR_motif"]
plt.figure(figsize=(7, 4))
ax = sns.lineplot(
    data=str_df,
    x="len_or_motif",
    y="count",
    hue="sample",
    style="indel_type",
    markers=True,
)
ax.set_xlabel("STR motif length (bp)")
ax.set_ylabel("Count")
ax.set_title("STR motif length spectrum")
plt.tight_layout()
plt.savefig(figures_dir / "spectrum_str_motif.png", dpi=300)
plt.savefig(figures_dir / "spectrum_str_motif.pdf")
plt.show()

In [ ]:
# Callable space per sample
callable_summary = (
    summary_df[["sample", "str_class", "total_callable"]]
    .sort_values(["sample", "str_class"])
)

plt.figure(figsize=(6, 4))
ax = sns.barplot(
    data=callable_summary,
    x="sample",
    y="total_callable",
    hue="str_class",
)
ax.set_ylabel("Callable bases")
ax.set_xlabel("Sample")
ax.set_title("Callable space by sample")
ax.ticklabel_format(style="sci", axis="y", scilimits=(0, 0))
plt.tight_layout()
plt.savefig(figures_dir / "callable_space.png", dpi=300)
plt.savefig(figures_dir / "callable_space.pdf")
plt.show()

In [ ]:
# Unique vs total rate QC
plt.figure(figsize=(6, 4))
ax = sns.scatterplot(
    data=summary_df,
    x="rate",
    y="unique_rate",
    hue="sample",
    style="str_class",
    s=80,
)
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("Total rate")
ax.set_ylabel("Unique-site rate")
ax.set_title("QC: unique vs total rate")
plt.tight_layout()
plt.savefig(figures_dir / "qc_unique_vs_total.png", dpi=300)
plt.savefig(figures_dir / "qc_unique_vs_total.pdf")
plt.show()

In [ ]:
# Validation: report any missing samples or files
if missing:
    print("Missing sample runs/files:")
    for item in missing:
        print(item)
else:
    print("All samples loaded successfully.")